In [ ]:
!pip install Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.0 MB/s eta 0:00:00


In [ ]:

import json
import time
import numpy as np
import pandas as pd
from itertools import product
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from groq import Groq
from google.colab import userdata



# Get a free Groq key: https://console.groq.com/keys
# Store in Colab Secrets (key icon, left sidebar) as GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    groq_client = Groq(api_key=GROQ_API_KEY)
    GROQ_LIVE = True
except Exception:
    GROQ_LIVE = False
    print("Groq key not found — API cells will show placeholder behavior.")

GROQ_MODEL = "llama-3.3-70b-versatile"

def call_groq(system: str, prompt: str) -> str:
    if not GROQ_LIVE:
        return "[placeholder — GROQ_API_KEY not configured]"
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
    )
    return response.choices[0].message.content

print(f"Setup complete. Groq live: {GROQ_LIVE}")

Setup complete. Groq live: True


In [ ]:
# --- What AutoML automates ---

AUTOML_PIPELINE = {
    "Step": [
        "Feature preprocessing",
        "Algorithm selection",
        "Hyperparameter tuning",
        "Pipeline assembly",
        "Cross-validation",
        "Ensemble construction",
    ],
    "Manual ML": [
        "Engineer by hand",
        "Trial and error",
        "GridSearch / RandomSearch",
        "Manual fit/transform chains",
        "Write CV loops",
        "Manually blend models",
    ],
    "AutoML": [
        "Automated imputation, scaling, encoding",
        "Search over model families",
        "Bayesian / evolutionary search",
        "Automatically chained",
        "Built-in",
        "Automated stacking",
    ],
}

df = pd.DataFrame(AUTOML_PIPELINE)
print(df.to_string(index=False))

                 Step                   Manual ML                                  AutoML
Feature preprocessing            Engineer by hand Automated imputation, scaling, encoding
  Algorithm selection             Trial and error              Search over model families
Hyperparameter tuning   GridSearch / RandomSearch          Bayesian / evolutionary search
    Pipeline assembly Manual fit/transform chains                   Automatically chained
     Cross-validation              Write CV loops                                Built-in
Ensemble construction       Manually blend models                      Automated stacking


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
SEARCH_SPACE = [
    ("LogisticRegression", LogisticRegression, {"C": [0.1, 1.0, 10.0], "max_iter": [200]}),
    ("RandomForest",       RandomForestClassifier, {"n_estimators": [50, 100], "max_depth": [None, 5]}),
    ("GradientBoosting",   GradientBoostingClassifier, {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1]}),
    ("SVM",                SVC, {"C": [0.1, 1.0], "kernel": ["rbf", "linear"]}),
    ("KNN",                KNeighborsClassifier, {"n_neighbors": [3, 5, 7]}),
]

In [ ]:

results = []
for name, Model, param_grid in SEARCH_SPACE:
    keys, values = zip(*param_grid.items())
    for combo in product(*values):
        params = dict(zip(keys, combo))
        pipe = Pipeline([("scaler", StandardScaler()), ("model", Model(**params))])
        cv_score = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy").mean()
        results.append({"model": name, "params": params, "cv_accuracy": round(cv_score, 4)})

results_df = pd.DataFrame(results).sort_values("cv_accuracy", ascending=False)
print(f"Configurations evaluated: {len(results_df)}\n")
print(results_df.head(10).to_string(index=False))

Configurations evaluated: 18

             model                                  params  cv_accuracy
LogisticRegression             {'C': 1.0, 'max_iter': 200}       0.9758
LogisticRegression            {'C': 10.0, 'max_iter': 200}       0.9758
               SVM          {'C': 0.1, 'kernel': 'linear'}       0.9736
               SVM             {'C': 1.0, 'kernel': 'rbf'}       0.9736
LogisticRegression             {'C': 0.1, 'max_iter': 200}       0.9714
               SVM          {'C': 1.0, 'kernel': 'linear'}       0.9714
      RandomForest {'n_estimators': 50, 'max_depth': None}       0.9626
               KNN                      {'n_neighbors': 3}       0.9604
               KNN                      {'n_neighbors': 5}       0.9604
               KNN                      {'n_neighbors': 7}       0.9560


In [ ]:
# --- Fit and evaluate the best configuration found ---

best = results_df.iloc[0]
ModelClass = dict((n, M) for n, M, _ in SEARCH_SPACE)[best["model"]]
best_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  ModelClass(**best["params"]))
])
best_pipe.fit(X_train, y_train)
test_acc = accuracy_score(y_test, best_pipe.predict(X_test))

print(f"Best model:       {best['model']}")
print(f"Best params:      {best['params']}")
print(f"CV accuracy:      {best['cv_accuracy']}")
print(f"Test accuracy:    {round(test_acc, 4)}")

Best model:       LogisticRegression
Best params:      {'C': 1.0, 'max_iter': 200}
CV accuracy:      0.9758
Test accuracy:    0.9737


In [ ]:
# --- When to use AutoML vs manual ML ---

DECISION_TABLE = {
    "Situation": [
        "Quick baseline needed",
        "Domain expertise available",
        "Interpretability required",
        "Large dataset, compute available",
        "Custom loss / architecture needed",
        "Time-constrained prototype",
        "Production model needing full control",
    ],
    "AutoML": ["✅", "⚠️", "⚠️", "✅", "❌", "✅", "⚠️"],
    "Manual ML": ["⚠️", "✅", "✅", "⚠️", "✅", "❌", "✅"],
}

df = pd.DataFrame(DECISION_TABLE)
print(df.to_string(index=False))
print("\n✅=preferred   ⚠️=possible   ❌ = not suitable")

# STUDENT TRY: add one more row to DECISION_TABLE for a use case from your domain
# and justify the AutoML vs Manual ML choice.

                            Situation AutoML Manual ML
                Quick baseline needed      ✅        ⚠️
           Domain expertise available     ⚠️         ✅
            Interpretability required     ⚠️         ✅
     Large dataset, compute available      ✅        ⚠️
    Custom loss / architecture needed      ❌         ✅
           Time-constrained prototype      ✅         ❌
Production model needing full control     ⚠️         ✅

✅=preferred   ⚠️=possible   ❌ = not suitable


In [ ]:
# --- What lm-evaluation-harness does ---
# lm-eval (EleutherAI) is the standard open-source framework for evaluating LLMs
# on standardised benchmarks. We simulate its evaluation loop here —
# the actual library runs identically but against full model APIs.

LM_EVAL_PIPELINE = {
    "Stage": [
        "1. Task loading",
        "2. Prompt formatting",
        "3. Model inference",
        "4. Output parsing",
        "5. Metric computation",
        "6. Results aggregation",
    ],
    "What happens": [
        "Load dataset (MMLU, HellaSwag, etc.) with splits and labels",
        "Format each example into a prompt using the task's template",
        "Send prompt to model, collect completion or log-probs",
        "Extract model's choice from raw output",
        "Compare prediction vs label → accuracy / F1 / etc.",
        "Average across examples, report per-task and overall",
    ],
}

df = pd.DataFrame(LM_EVAL_PIPELINE)
print(df.to_string(index=False))

                 Stage                                                What happens
       1. Task loading Load dataset (MMLU, HellaSwag, etc.) with splits and labels
  2. Prompt formatting Format each example into a prompt using the task's template
    3. Model inference       Send prompt to model, collect completion or log-probs
     4. Output parsing                      Extract model's choice from raw output
 5. Metric computation          Compare prediction vs label → accuracy / F1 / etc.
6. Results aggregation        Average across examples, report per-task and overall


In [ ]:
SAMPLE_TASK = [
    {
        "question": "What is the capital of France?",
        "choices": ["A. London", "B. Berlin", "C. Paris", "D. Madrid"],
        "answer": "C",
    },
    {
        "question": "Which gas do plants absorb during photosynthesis?",
        "choices": ["A. Oxygen", "B. Nitrogen", "C. Hydrogen", "D. Carbon dioxide"],
        "answer": "D",
    },
    {
        "question": "What is 15% of 200?",
        "choices": ["A. 20", "B. 25", "C. 30", "D. 35"],
        "answer": "C",
    },
    {
        "question": "Who wrote Romeo and Juliet?",
        "choices": ["A. Dickens", "B. Shakespeare", "C. Austen", "D. Marlowe"],
        "answer": "B",
    },
    {
        "question": "What is the powerhouse of the cell?",
        "choices": ["A. Nucleus", "B. Ribosome", "C. Mitochondria", "D. Golgi apparatus"],
        "answer": "C",
    },
]

In [ ]:
EVAL_SYSTEM = """You are taking a multiple choice exam.
Read the question and choices carefully.
Respond with ONLY the letter of the correct answer (A, B, C, or D).
Nothing else."""

In [ ]:
from os.path import join
def evaluate_task(task_examples:list,model_fn) -> dict:
  predictions=[]
  labels=[]
  #"question": "What is the powerhouse of the cell?",
  for i in task_examples:
    prompt = f"Question: {i['question']}\n"+ "\n".join(i["choices"])
    raw = model_fn(EVAL_SYSTEM, prompt).strip().upper()
    pred = raw[0] if raw and raw[0] in "ABCD" else "X"
    predictions.append(pred)
    labels.append(i["answer"])
    print(f"  Q: {i['question'][:50]:<50} Pred: {pred}  Label: {i['answer']}  {'✅' if pred == i['answer'] else '❌'}")
  accuracy = sum(p == l for p, l in zip(predictions, labels)) / len(labels)
  return {
      "accuracy": round(accuracy, 3),
      "n": len(labels),
      "correct": sum(p == l for p, l in zip(predictions, labels))
  }


In [ ]:
result = evaluate_task(SAMPLE_TASK, call_groq)
print(f"\nAccuracy: {result['correct']}/{result['n']} = {result['accuracy']}")

  Q: What is the capital of France?                     Pred: C  Label: C  ✅
  Q: Which gas do plants absorb during photosynthesis?  Pred: D  Label: D  ✅
  Q: What is 15% of 200?                                Pred: C  Label: C  ✅
  Q: Who wrote Romeo and Juliet?                        Pred: B  Label: B  ✅
  Q: What is the powerhouse of the cell?                Pred: C  Label: C  ✅

Accuracy: 5/5 = 1.0


In [ ]:
MMLU_SAMPLES = {
    "high_school_mathematics": [
        {
            "question": "If f(x) = 2x² + 3x - 5, what is f(2)?",
            "choices": ["A. 7", "B. 9", "C. 11", "D. 13"],
            "answer": "B",
        },
        {
            "question": "What is the derivative of sin(x)?",
            "choices": ["A. -cos(x)", "B. cos(x)", "C. -sin(x)", "D. tan(x)"],
            "answer": "B",
        },
    ],
    "medical_genetics": [
        {
            "question": "Which type of mutation causes sickle cell anaemia?",
            "choices": ["A. Frameshift", "B. Nonsense", "C. Missense", "D. Silent"],
            "answer": "C",
        },
        {
            "question": "What does PCR stand for?",
            "choices": ["A. Polymerase chain reaction", "B. Protein chain replication", "C. Plasma cell ratio", "D. Primary chromosomal recombination"],
            "answer": "A",
        },
    ],
    "world_history": [
        {
            "question": "In which year did World War II end?",
            "choices": ["A. 1943", "B. 1944", "C. 1945", "D. 1946"],
            "answer": "C",
        },
        {
            "question": "The Treaty of Versailles was signed after which war?",
            "choices": ["A. World War II", "B. World War I", "C. Franco-Prussian War", "D. Crimean War"],
            "answer": "B",
        },
    ],
}


In [ ]:
print("=== MMLU evaluation by subject ===\n")
subject_scores = {}
for subject, examples in MMLU_SAMPLES.items():
    print(f"Subject: {subject}")
    result = evaluate_task(examples, call_groq)
    subject_scores[subject] = result["accuracy"]
    print()

print("--- MMLU subject scores ---")
for subject, score in subject_scores.items():
    print(f"  {subject:<30} {score:.2f}")
print(f"\n  {'Macro average':<30} {np.mean(list(subject_scores.values())):.2f}")

=== MMLU evaluation by subject ===

Subject: high_school_mathematics
  Q: If f(x) = 2x² + 3x - 5, what is f(2)?              Pred: D  Label: B  ❌
  Q: What is the derivative of sin(x)?                  Pred: B  Label: B  ✅

Subject: medical_genetics
  Q: Which type of mutation causes sickle cell anaemia? Pred: C  Label: C  ✅
  Q: What does PCR stand for?                           Pred: A  Label: A  ✅

Subject: world_history
  Q: In which year did World War II end?                Pred: C  Label: C  ✅
  Q: The Treaty of Versailles was signed after which wa Pred: B  Label: B  ✅

--- MMLU subject scores ---
  high_school_mathematics        0.50
  medical_genetics               1.00
  world_history                  1.00

  Macro average                  0.83


In [ ]:
HELLASWAG_SAMPLES = [
    {
        "question": "A woman is outside with a bucket and a dog. The dog is running around trying to avoid a bath. She...",
        "choices": [
            "A. rinses the bucket out and sets it on the floor.",
            "B. gets the dog wet, then soaps and rinses it.",
            "C. starts to blow dry the dog's fur.",
            "D. brushes the dog's fur and walks away.",
        ],
        "answer": "B",
    },
    {
        "question": "A man is making a sandwich. He lays two slices of bread on the counter. He...",
        "choices": [
            "A. puts the bread in the microwave for 5 minutes.",
            "B. adds fillings between the slices and presses them together.",
            "C. throws one slice in the bin and eats the other plain.",
            "D. pours a glass of water over the bread.",
        ],
        "answer": "B",
    },
    {
        "question": "Someone is cooking pasta. The water is boiling. They...",
        "choices": [
            "A. add pasta to the boiling water and stir.",
            "B. turn off the heat and let the water cool.",
            "C. pour the hot water into a glass to drink.",
            "D. put raw vegetables directly on the stove.",
        ],
        "answer": "A",
    },
    {
        "question": "A person is wrapping a birthday gift. They place the item in a box. They...",
        "choices": [
            "A. label the box with a shipping address.",
            "B. cover the box with wrapping paper and add a bow.",
            "C. fill the box with water.",
            "D. put the box in the refrigerator.",
        ],
        "answer": "B",
    },
]


In [ ]:
hellaswag_result = evaluate_task(HELLASWAG_SAMPLES, call_groq)
print(f"\nAccuracy: {hellaswag_result['correct']}/{hellaswag_result['n']} = {hellaswag_result['accuracy']}")

  Q: A woman is outside with a bucket and a dog. The do Pred: B  Label: B  ✅
  Q: A man is making a sandwich. He lays two slices of  Pred: B  Label: B  ✅
  Q: Someone is cooking pasta. The water is boiling. Th Pred: A  Label: A  ✅
  Q: A person is wrapping a birthday gift. They place t Pred: B  Label: B  ✅

Accuracy: 4/4 = 1.0


In [ ]:
# --- MMLU vs HellaSwag: what each benchmark actually measures ---

BENCHMARK_COMPARISON = {
    "Benchmark": ["MMLU", "HellaSwag"],
    "What it tests": [
        "Domain knowledge across 57 subjects",
        "Commonsense reasoning and situational understanding",
    ],
    "Format": [
        "4-choice MCQ, factual knowledge",
        "4-choice sentence completion, plausibility judgment",
    ],
    "Failure mode": [
        "Model can score well by memorising training data",
        "Model can score well by statistical pattern matching",
    ],
    "Primary use": [
        "Knowledge breadth at a point in time",
        "Reasoning generalisation",
    ],
}

df = pd.DataFrame(BENCHMARK_COMPARISON)
print(df.to_string(index=False))

Benchmark                                       What it tests                                              Format                                         Failure mode                          Primary use
     MMLU                 Domain knowledge across 57 subjects                     4-choice MCQ, factual knowledge     Model can score well by memorising training data Knowledge breadth at a point in time
HellaSwag Commonsense reasoning and situational understanding 4-choice sentence completion, plausibility judgment Model can score well by statistical pattern matching             Reasoning generalisation


In [ ]:
# --- Data contamination: when benchmark data leaks into training ---

CONTAMINATION_SCENARIOS = [
    {
        "scenario": "Benchmark exact match",
        "description": "Training data contains verbatim MMLU questions and answers",
        "signal": "Near-perfect accuracy on MMLU but poor performance on novel questions",
        "detection": "Test on held-out benchmark versions or perturbed question variants",
    },
    {
        "scenario": "Near-duplicate overlap",
        "description": "Paraphrased versions of benchmark questions appear in training data",
        "signal": "High accuracy on benchmark, poor accuracy on rephrased versions of same questions",
        "detection": "n-gram overlap analysis between training corpus and benchmark",
    },
    {
        "scenario": "Answer distribution leak",
        "description": "Model has seen answer keys but not questions — learns answer patterns",
        "signal": "Model outputs correct letter (A/B/C/D) but wrong reasoning",
        "detection": "Ask model to explain reasoning — surface-level correct, explanation wrong",
    },
    {
        "scenario": "Web crawl contamination",
        "description": "Benchmark leaderboard pages or answer discussions scraped into training",
        "signal": "Model scores well on public benchmarks, poorly on private held-out sets",
        "detection": "Evaluate on private benchmarks not publicly discussed online",
    },
]

for s in CONTAMINATION_SCENARIOS:
    print(f"Scenario:   {s['scenario']}")
    print(f"What:       {s['description']}")
    print(f"Signal:     {s['signal']}")
    print(f"Detection:  {s['detection']}\n")

Scenario:   Benchmark exact match
What:       Training data contains verbatim MMLU questions and answers
Signal:     Near-perfect accuracy on MMLU but poor performance on novel questions
Detection:  Test on held-out benchmark versions or perturbed question variants

Scenario:   Near-duplicate overlap
What:       Paraphrased versions of benchmark questions appear in training data
Signal:     High accuracy on benchmark, poor accuracy on rephrased versions of same questions
Detection:  n-gram overlap analysis between training corpus and benchmark

Scenario:   Answer distribution leak
What:       Model has seen answer keys but not questions — learns answer patterns
Signal:     Model outputs correct letter (A/B/C/D) but wrong reasoning
Detection:  Ask model to explain reasoning — surface-level correct, explanation wrong

Scenario:   Web crawl contamination
What:       Benchmark leaderboard pages or answer discussions scraped into training
Signal:     Model scores well on public benchmarks, 

In [ ]:
ORIGINAL_QUESTIONS = [
    {
        "question": "What is the capital of France?",
        "choices": ["A. London", "B. Berlin", "C. Paris", "D. Madrid"],
        "answer": "C",
    },
    {
        "question": "What gas do plants absorb during photosynthesis?",
        "choices": ["A. Oxygen", "B. Nitrogen", "C. Hydrogen", "D. Carbon dioxide"],
        "answer": "D",
    },
]

In [ ]:
PERTURBED_QUESTIONS = [
    {
        "question": "Which city serves as the national capital of France?",
        "choices": ["A. Madrid", "B. Paris", "C. Berlin", "D. London"],
        "answer": "B",
    },
    {
        "question": "During photosynthesis, which atmospheric gas is taken up by plants?",
        "choices": ["A. Carbon dioxide", "B. Hydrogen", "C. Nitrogen", "D. Oxygen"],
        "answer": "A",
    },
]

In [ ]:
print("=== Original questions ===")
orig_result = evaluate_task(ORIGINAL_QUESTIONS, call_groq)
print(f"Accuracy: {orig_result['accuracy']}\n")

print("=== Perturbed questions (same knowledge, different form) ===")
perturbed_result = evaluate_task(PERTURBED_QUESTIONS, call_groq)
print(f"Accuracy: {perturbed_result['accuracy']}\n")

drop = orig_result["accuracy"] - perturbed_result["accuracy"]
print(f"Accuracy drop: {drop:.2f}")
print("A large drop (>0.15) on perturbed questions is a contamination signal.")
print("A robust model should score similarly on both.")

=== Original questions ===
  Q: What is the capital of France?                     Pred: C  Label: C  ✅
  Q: What gas do plants absorb during photosynthesis?   Pred: D  Label: D  ✅
Accuracy: 1.0

=== Perturbed questions (same knowledge, different form) ===
  Q: Which city serves as the national capital of Franc Pred: B  Label: B  ✅
  Q: During photosynthesis, which atmospheric gas is ta Pred: A  Label: A  ✅
Accuracy: 1.0

Accuracy drop: 0.00
A large drop (>0.15) on perturbed questions is a contamination signal.
A robust model should score similarly on both.


In [ ]:
# --- End-to-end: AutoML + LLM evaluation comparison ---
# Shows how the evaluation principles from this session apply
# whether you're evaluating an AutoML model or an LLM.

print("Evaluation principles: AutoML vs LLM\n")

EVAL_COMPARISON = {
    "Dimension": [
        "Model selection",
        "Evaluation metric",
        "Overfitting risk",
        "Contamination risk",
        "Benchmark",
        "Human eval",
    ],
    "AutoML": [
        "Cross-validated accuracy on holdout",
        "Accuracy / F1 / AUC on test set",
        "CV score >> test score",
        "Train/test leakage via preprocessing on full data",
        "UCI datasets, Kaggle benchmarks",
        "Domain expert review on edge cases",
    ],
    "LLM Evaluation": [
        "Accuracy on MMLU, HellaSwag, etc.",
        "Accuracy, BLEU, ROUGE, faithfulness",
        "Benchmark-specific fine-tuning",
        "Benchmark data in pre-training corpus",
        "MMLU, HellaSwag, BIG-Bench, HELM",
        "Human raters on 5-dimension rubric",
    ],
}

df = pd.DataFrame(EVAL_COMPARISON)
print(df.to_string(index=False))

# STUDENT TRY: pick one row from this table and describe a real scenario
# where that failure mode caused a misleading evaluation result.

Evaluation principles: AutoML vs LLM

         Dimension                                            AutoML                        LLM Evaluation
   Model selection               Cross-validated accuracy on holdout     Accuracy on MMLU, HellaSwag, etc.
 Evaluation metric                   Accuracy / F1 / AUC on test set   Accuracy, BLEU, ROUGE, faithfulness
  Overfitting risk                            CV score >> test score        Benchmark-specific fine-tuning
Contamination risk Train/test leakage via preprocessing on full data Benchmark data in pre-training corpus
         Benchmark                   UCI datasets, Kaggle benchmarks      MMLU, HellaSwag, BIG-Bench, HELM
        Human eval                Domain expert review on edge cases    Human raters on 5-dimension rubric
